# Ruby AI Pull Request Reviewer

This notebook runs the complete pipeline:

GitHub PR → diff parsing → repository checkout → RuboCop → RAG/FAISS → OpenAI → structured review → merged findings.


## 0. Configuration

Before running this notebook, make sure your project contains:

```text
review-ai/
├── notebook.ipynb
├── requirements.txt
├── .env
├── knowledge/
│   ├── rails.md
│   ├── security.md
│   ├── performance.md
│   ├── clean_code.md
│   └── testing.md
└── src/
    ├── __init__.py
    ├── config.py
    ├── github.py
    ├── parser.py
    ├── rag.py
    ├── rubocop.py
    ├── openai_client.py
    ├── reviewer.py
    ├── schema.py
    └── aggregator.py
```

Your `.env` should contain:

```text
OPENAI_API_KEY=your_key_here
```

In [ ]:
from pathlib import Path
import sys
import os

# Make the project root importable when this notebook is run from the project directory.
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

## 1. Imports

Run this after installing `requirements.txt`.

In [ ]:
from src.github import (
    download_pr,
    clone_pr_repository,
    get_pr_metadata,
    parse_pr_url,
)

from src.parser import parse_diff
from src.rag import load_vector_db, retrieve
from src.rubocop import run_rubocop, format_offenses
from src.reviewer import review
from src.aggregator import rubocop_to_issues, merge_review

print("Imports OK")

## 2. Choose the Pull Request

For the first demo, use a public GitHub repository. Private repositories require GitHub authentication.

In [ ]:
PR_URL = "https://github.com/renzodiaz/notes-api/pull/4"

print("PR:", PR_URL)

## 3. Validate the PR URL

In [ ]:
pr_info = parse_pr_url(PR_URL)

print("Owner:", pr_info["owner"])
print("Repository:", pr_info["repo"])
print("PR number:", pr_info["number"])

## 4. Get PR metadata from GitHub

In [ ]:
metadata = get_pr_metadata(PR_URL)

for key, value in metadata.items():
    print(f"{key}: {value}")

## 5. Download the PR diff

The diff is the primary input for understanding what changed.

In [ ]:
diff = download_pr(PR_URL)

print(f"Downloaded {len(diff):,} characters.")
print(f"Diff lines: {len(diff.splitlines()):,}")

## 6. Inspect the raw diff

This is optional, but useful while developing.

In [ ]:
print(diff[:8000])

## 7. Parse the diff

The parser identifies changed files, Ruby files, production files, tests, and diff hunks.

In [ ]:
parsed = parse_diff(diff)

print("Changed files:", len(parsed["changed_files"]))
print("Ruby files:", len(parsed["ruby_files"]))
print("Production Ruby files:", len(parsed["production_files"]))
print("Test files:", len(parsed["test_files"]))

## 8. Show changed files

In [ ]:
for file in parsed["changed_files"]:
    print(
        f"{file['filename']} | "
        f"Ruby={file['is_ruby']} | "
        f"Test={file['is_test']} | "
        f"Hunks={len(file['hunks'])}"
    )

## 9. Show the Ruby code being reviewed

The reviewer uses added Ruby code plus removed code and surrounding hunk context.

In [ ]:
changed_code = parsed["added_ruby_code"]

print(changed_code)

## 10. Build the RAG vector database

Markdown engineering guidelines are chunked, embedded with Hugging Face, and stored in FAISS.

In [ ]:
db = load_vector_db()

print("RAG vector database ready.")

## 11. Test RAG retrieval

This cell is shows we are using retrieving engineering knowledge.

In [ ]:
query = changed_code
retrieved_documents = retrieve(db, query)

print(f"Retrieved {len(retrieved_documents)} knowledge chunks.\n")

for index, document in enumerate(retrieved_documents, start=1):
    print("=" * 70)
    print(f"RESULT {index}")
    print("SOURCE:", document.metadata.get("source"))
    print(document.page_content[:1500])
            

## 12. Clone the PR repository

RuboCop needs the actual source tree, not just the diff.

In [ ]:
repo_path = clone_pr_repository(PR_URL)

print("Repository cloned to:")
print(repo_path)

## 13. Determine which Ruby files RuboCop should inspect

In [ ]:
ruby_files = [
    file["filename"]
    for file in parsed["ruby_files"]
]

print("Ruby files:")
for filename in ruby_files:
    print("-", filename)

## 14. Run RuboCop

RuboCop provides deterministic static-analysis findings.

In [ ]:
offenses = run_rubocop(
    project_path=repo_path,
    files=ruby_files,
)

print(f"RuboCop found {len(offenses)} offenses.")

## 15. Display RuboCop findings

In [ ]:
rubocop_context = format_offenses(offenses)

print(rubocop_context)

## 16. Convert RuboCop findings to the common review schema

In [ ]:
static_issues = rubocop_to_issues(offenses)

print(f"Normalized static issues: {len(static_issues)}")

for issue in static_issues:
    print(
        f"[{issue.severity.upper()}] "
        f"{issue.title} - "
        f"{issue.file}:{issue.line}"
    )

## 17. Run the AI review

The LLM receives the changed Ruby code, retrieved RAG context, and RuboCop findings.

In [ ]:
ai_result = review(
    changed_code=changed_code,
    db=db,
    rubocop_context=rubocop_context,
)

print("AI review completed.")

## 18. Inspect the AI result

In [ ]:
print("SUMMARY")
print(ai_result.summary)
print()
print("SCORE:", ai_result.score)
print("ISSUES:", len(ai_result.issues))
print()

for issue in ai_result.issues:
    print(
        f"[{issue.severity.upper()}] "
        f"{issue.title}"
    )
    print("Type:", issue.type)
    print("Category:", issue.category)
    print("File:", issue.file)
    print("Line:", issue.line)
    print("Explanation:", issue.explanation)
    print("Recommendation:", issue.recommendation)
    print("Evidence:", ", ".join(issue.evidence))
    print("-" * 70)

## 19. Merge AI + RuboCop findings

The aggregator removes obvious duplicate findings.

In [ ]:
final_result = merge_review(
    ai_result=ai_result,
    static_issues=static_issues,
)

print("Final review assembled.")

## 20. Final human-readable report

In [ ]:
print("=" * 80)
print("RUBY AI PULL REQUEST REVIEW")
print("=" * 80)
            
print()
print("PR:", PR_URL)
print("Repository:", f"{pr_info['owner']}/{pr_info['repo']}")
print("PR number:", pr_info["number"])
print("Changed files:", len(parsed["changed_files"]))
print("Ruby files:", len(parsed["ruby_files"]))
print("Test files:", len(parsed["test_files"]))
print("RuboCop offenses:", len(offenses))
            
print()
print("SCORE:", f"{final_result.score}/100")
print()
print("SUMMARY")
print(final_result.summary)
print()
print("ISSUES")
print("=" * 80)

for index, issue in enumerate(final_result.issues, start=1):
    print(f"\n{index}. [{issue.severity.upper()}] {issue.title}")
    print(f"   Category: {issue.category}")
    print(f"   File: {issue.file}")
    print(f"   Line: {issue.line}")
    print(f"   Explanation: {issue.explanation}")
    print(f"   Recommendation: {issue.recommendation}")
    print(f"   Evidence: {', '.join(issue.evidence)}")

print()
print("POSITIVE FINDINGS")
for finding in final_result.positive_findings:
    print("-", finding)

## 21. Export the final result as JSON

This will later be useful for the evaluation benchmark and Gradio UI.

In [ ]:
result_json = final_result.model_dump_json(indent=2)
print(result_json)

## 22. Simple statistics for this PR

In [ ]:
severity_counts = {}
category_counts = {}

for issue in final_result.issues:
    severity_counts[issue.severity] = severity_counts.get(issue.severity, 0) + 1
    category_counts[issue.category] = category_counts.get(issue.category, 0) + 1

print("Severity distribution:")
for severity, count in sorted(severity_counts.items()):
    print(f"- {severity}: {count}")

print()
print("Category distribution:")
for category, count in sorted(category_counts.items()):
    print(f"- {category}: {count}")

## Pipeline complete

The current system is:

```text
GitHub PR
   │
   ├── .diff ──→ Diff Parser ──→ Changed Ruby Code
   │                                  │
   │                                  ▼
   │                              RAG / FAISS
   │                                  │
   │                                  ▼
   └── Repository ──→ RuboCop ─────→ OpenAI
                                      │
                                      ▼
                               Structured Review
                                      │
                                      ▼
                                  Final Report
```

Next milestone: create a controlled benchmark with intentionally flawed Ruby PRs and calculate precision, recall, F1, false-positive rate, and review time.

# Run Cases

In [ ]:
from src.evaluator import (
    load_cases,
    load_expected_findings,
    run_case,
    extract_issue_types,
)

In [ ]:
cases = load_cases()

print(
    f"Loaded {len(cases)} benchmark cases."
)

## Run Benchmark

In [ ]:
expected = load_expected_findings()

benchmark_results = []

for case in cases:

    print(
        f"Running {case['case_id']}..."
    )

    result = run_case(
        case,
        db,
    )

    detected = extract_issue_types(
        result
    )

    expected_types = {
        item["type"]
        for item in expected[
            case["case_id"]
        ]
    }

    benchmark_results.append(
        {
            "case_id": case["case_id"],
            "expected": expected_types,
            "detected": detected,
        }
    )

## Display Results

In [ ]:
for result in benchmark_results:

    print("=" * 60)

    print(
        "CASE:",
        result["case_id"],
    )

    print(
        "EXPECTED:",
        result["expected"],
    )

    print(
        "DETECTED:",
        result["detected"],
    )

## Evaluation Metrics

Score the benchmark with precision, recall, F1, success rate (per-case pass/fail), reliability (runs without exceptions), and average latency. Run this before the demo to get real numbers for the evaluation report.

In [ ]:
from src.evaluator import run_benchmark, compute_metrics
import json
from pathlib import Path

benchmark_results = run_benchmark(db)
metrics = compute_metrics(benchmark_results)

print(f"Cases: {metrics['total_cases']}")
print(f"Precision: {metrics['precision']}")
print(f"Recall: {metrics['recall']}")
print(f"F1: {metrics['f1']}")
print(f"Success rate (per-case pass/fail): {metrics['success_rate']}")
print(f"Reliability (no exceptions): {metrics['reliability']}")
print(f"Avg latency per case: {metrics['avg_latency_seconds']}s")
print()

for case in metrics["per_case"]:
    status = "PASS" if case["passed"] else "FAIL"
    print(f"[{status}] {case['case_id']:<28} expected={sorted(case['expected'])} detected={sorted(case['detected'])} ({case['elapsed_seconds']:.2f}s)")
    if case["error"]:
        print(f"         ERROR: {case['error']}")

# Persist the report so it can be pasted into the evaluation deliverable.
report_path = Path("reports/evaluation_metrics.json")
report_path.parent.mkdir(parents=True, exist_ok=True)

serializable = {
    **{k: v for k, v in metrics.items() if k != "per_case"},
    "per_case": [
        {**c, "expected": sorted(c["expected"]), "detected": sorted(c["detected"])}
        for c in metrics["per_case"]
    ],
}

report_path.write_text(json.dumps(serializable, indent=2))
print(f"\nSaved: {report_path}")

In [ ]:
from pathlib import Path

for path in sorted(Path("src").glob("*.py")):
    print("\n" + "=" * 80)
    print(f"FILE: {path}")
    print("=" * 80)
    print(path.read_text())